In [ ]:
%cd ../..
import os
import torch
import numpy as np
from omegaconf import OmegaConf
from einops import rearrange
import pandas as pd
from glob import glob
import SimpleITK as sitk

from dinov2.inference import build_model, view_volume
from evaluation import *

In [ ]:
def crop_pos_mdh(path, posx, posy, posz, diameter, pad=10):
    image_obj = sitk.ReadImage(path)
    image = sitk.GetArrayFromImage(image_obj)
    image = image.clip(-1000, 1900)

    D,W,H = image.shape
    
    origin = np.array(image_obj.GetOrigin())
    spacing = np.array(image_obj.GetSpacing())
    spacing = spacing[::-1]

    radius = [round(diameter/2/spacing[0]), round(diameter/2/spacing[1]) + pad, round(diameter/2/spacing[2]) + pad]

    coordx = round((posx - origin[0])/spacing[2])
    coordy = round((posy - origin[1])/spacing[1])
    coordz = round((posz - origin[2])/spacing[0])

    xmin, xmax = coordx - radius[2], coordx + radius[2]
    ymin, ymax = coordy - radius[1], coordy + radius[1]
    zmin, zmax = coordz - radius[0], coordz + radius[0]

    xmin = max(0, xmin)
    ymin = max(0, ymin)
    zmin = max(0, zmin)

    xmax = min(xmax, D)
    ymax = min(ymax, W)
    zmax = min(zmax,H)

    slice_obj = (slice(zmin, zmax, None), slice(ymin,ymax, None), slice(xmin, xmax, None))
    cropped_img = image[slice_obj]
    cropped_img = torch.from_numpy(cropped_img).float()
    
    return cropped_img, spacing


In [ ]:
dataset_root = "/scratch/VM/radio-foundation/datasets-recent/LUNA16"
img_paths = glob(os.path.join(dataset_root,"subset*/**/*.mhd"), recursive=True)
id_to_path = {p.split("/")[-1].replace(".mhd", ""): p for p in img_paths}
len(img_paths)

In [ ]:
candidates_df = pd.read_csv(os.path.join(dataset_root, "candidates.csv"))
candidatesv2_df = pd.read_csv(os.path.join(dataset_root, "candidates_V2.csv"))
annotations_df = pd.read_csv(os.path.join(dataset_root, "annotations.csv"))
annotations_df

In [ ]:
seriesuid, coordX, coordY, coordZ, diameter = annotations_df.iloc[0]
path = id_to_path[seriesuid]
path

In [ ]:
img, spacing = crop_pos_mdh(path, coordX, coordY, coordZ, diameter, pad=20)
view_volume(img, spacing)

In [ ]:
config_path = "/home/48078029W/projects/radio-foundation/runs/base10pat/config.yaml"
checkpoint_path = "/home/48078029W/projects/radio-foundation/runs/base10pat/eval/f/teacher_checkpoint.pth"

device = torch.device("cuda")

config = OmegaConf.load(config_path)
model, autocast_ctx = build_model(checkpoint_path, config, img_size=504, device=device)

In [ ]:
img_size = 98
patch_size = 14
channels = 10
patch_dim = img_size // patch_size

test_img = img
test_img = test_img.unsqueeze(0)
test_img = torch.nn.functional.interpolate(
    test_img.unsqueeze(0), size=(channels, img_size, img_size), mode="trilinear"
).squeeze(0)

with torch.inference_mode():
    with autocast_ctx():
        features = model.forward_features(test_img.cuda())
features = {k: v.cpu() for k, v in features.items() if isinstance(v, torch.Tensor)}
patch_features = features["x_norm_patchtokens"]
patch_features = rearrange(patch_features, "1 (x y) d -> x y d", x=patch_dim, y=patch_dim)
patch_features.shape

In [ ]:
plot_patch_similarity(patch_features, test_img, ref_x = 3, ref_y = 3, thresh=0.0)

In [ ]:
plot_feature_map(patch_features, test_img, (0,1,2))
